# TSLANet: Time Series Lightweight Adaptive Attention Network

This notebook demonstrates how to use TSLANet for time series classification. We'll cover:

1. Setting up the environment and importing necessary libraries
2. Loading and preprocessing time series data
3. Defining the TSLANet model architecture
4. Training the model on a classification task
5. Evaluating model performance and analyzing results
6. Exploring the spectral behavior of the Adaptive Spectral Block (ASB)

## Setup

First, let's import the necessary packages and set up the environment:

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

# Add the parent directory to path for importing from src
module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# Set style for plots
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

Now let's import the TSLANet model and related utilities:

In [ ]:
from src.models.tslanet.model import TSLANet, TSLANetPretraining
from src.data.loader import TimeSeriesDataset, create_dataloaders
from src.utils.utils import (
    plot_confusion_matrix,
    plot_training_curves,
    visualize_spectral_attention,
)
from src.training.trainer import generate_classification_report

## 1. Data Loading and Preparation

For this demo, we'll use the ECG200 dataset from the UCR/UEA Time Series Archive, which is a binary classification task for distinguishing between normal and abnormal heartbeats.

First, let's define a function to load the UCR dataset:

In [ ]:
def load_ucr_dataset(dataset_name):
    """
    Load a UCR dataset. This is a simplified version that assumes the data
    is already downloaded and formatted as numpy arrays.
    """
    # Path to the UCR datasets
    base_path = os.path.join("..", "data", "UCR")

    # Load training and test data
    X_train = np.loadtxt(
        os.path.join(base_path, dataset_name, f"{dataset_name}_TRAIN.tsv")
    )
    X_test = np.loadtxt(
        os.path.join(base_path, dataset_name, f"{dataset_name}_TEST.tsv")
    )

    # Split into features and labels
    y_train = X_train[:, 0].astype(int)
    X_train = X_train[:, 1:]

    y_test = X_test[:, 0].astype(int)
    X_test = X_test[:, 1:]

    # Add channel dimension and convert to PyTorch tensors
    X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
    X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

    # Get unique class labels and remap them to start from 0
    unique_labels = np.unique(np.concatenate([y_train, y_test]))
    label_map = {label: i for i, label in enumerate(unique_labels)}

    y_train = np.array([label_map[label] for label in y_train])
    y_test = np.array([label_map[label] for label in y_test])

    return X_train, y_train, X_test, y_test, list(label_map.keys())

Now let's load the ECG200 dataset:

In [ ]:
try:
    # Try to load the UCR dataset
    X_train, y_train, X_test, y_test, original_class_names = load_ucr_dataset("ECG200")
    print(f"Loaded ECG200 dataset with shapes:")
    print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
    print(f"Original class names: {original_class_names}")

    # Create class names for display
    class_names = [
        f"Class {i} (Original: {name})" for i, name in enumerate(original_class_names)
    ]
    num_classes = len(class_names)

except FileNotFoundError:
    print("UCR dataset not found. Using synthetic data for demonstration instead.")
    # Create synthetic data for demo purposes
    seq_len = 96
    num_classes = 2
    class_names = ["Normal", "Abnormal"]

    # Generate random time series data
    np.random.seed(42)
    n_samples = 200

    # Class 0: sine wave with noise
    X_class0 = np.zeros((n_samples // 2, 1, seq_len))
    t = np.linspace(0, 2 * np.pi, seq_len)
    for i in range(n_samples // 2):
        X_class0[i, 0, :] = np.sin(t + np.random.normal(0, 0.1)) + np.random.normal(
            0, 0.1, seq_len
        )

    # Class 1: sine wave with spike and noise
    X_class1 = np.zeros((n_samples // 2, 1, seq_len))
    for i in range(n_samples // 2):
        X_class1[i, 0, :] = np.sin(t + np.random.normal(0, 0.1))
        # Add spike
        spike_pos = np.random.randint(seq_len // 3, 2 * seq_len // 3)
        X_class1[i, 0, spike_pos : spike_pos + 5] += 2.0
        X_class1[i, 0, :] += np.random.normal(0, 0.1, seq_len)

    # Combine classes
    X = np.vstack([X_class0, X_class1])
    y = np.array([0] * (n_samples // 2) + [1] * (n_samples // 2))

    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    print(f"Created synthetic dataset with shapes:")
    print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

Let's visualize a few examples from each class:

In [ ]:
plt.figure(figsize=(12, 6))

for i, class_idx in enumerate(range(num_classes)):
    # Get examples from this class
    indices = np.where(y_train == class_idx)[0][
        :3
    ]  # Get first 3 examples of this class

    for j, idx in enumerate(indices):
        plt.subplot(num_classes, 3, i * 3 + j + 1)
        plt.plot(X_train[idx, 0])
        plt.title(f"{class_names[class_idx]} (Sample {j + 1})")
        plt.tight_layout()

plt.suptitle("Example Time Series from Each Class", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Prepare DataLoaders

Now let's create the PyTorch datasets and dataloaders:

In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

# Create datasets
train_dataset = TimeSeriesDataset(X_train_tensor, y_train_tensor, normalize=True)
test_dataset = TimeSeriesDataset(X_test_tensor, y_test_tensor, normalize=True)

# Create dataloaders with validation split
batch_size = 32
train_loader, val_loader, test_loader = create_dataloaders(
    train_dataset, test_dataset, batch_size=batch_size, val_split=0.2, num_workers=0
)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# Get sample batch for model configuration
sample_batch = next(iter(train_loader))
x_sample, y_sample = sample_batch
print(f"Sample batch shapes: x={x_sample.shape}, y={y_sample.shape}")

# Extract dataset properties
seq_len = x_sample.shape[2]
num_channels = x_sample.shape[1]
print(f"Sequence length: {seq_len}")
print(f"Number of channels: {num_channels}")

## 2. Building and Training the TSLANet Model

Now let's define a custom Lightning training wrapper for tracking metrics and visualization:

In [ ]:
class TSLANetTrainingWrapper(L.LightningModule):
    def __init__(self, model, learning_rate=1e-3):
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate

        # For tracking metrics
        self.train_losses = []
        self.val_losses = []
        self.train_accs = []
        self.val_accs = []
        self.current_epoch_train_loss = []
        self.current_epoch_train_acc = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.model.criterion(logits, y)
        acc = (logits.argmax(dim=-1) == y).float().mean()

        # Log metrics
        self.log("train_loss", loss)
        self.log("train_acc", acc)

        # Track for plotting
        self.current_epoch_train_loss.append(loss.item())
        self.current_epoch_train_acc.append(acc.item())

        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.model.criterion(logits, y)
        acc = (logits.argmax(dim=-1) == y).float().mean()

        # Log metrics
        self.log("val_loss", loss)
        self.log("val_acc", acc)

        return {"val_loss": loss, "val_acc": acc}

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.model.criterion(logits, y)
        acc = (logits.argmax(dim=-1) == y).float().mean()
        f1 = self.model.f1(logits, y)

        # Log metrics
        self.log("test_loss", loss)
        self.log("test_acc", acc)
        self.log("test_f1", f1)

        return {"test_loss": loss, "test_acc": acc, "test_f1": f1}

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.learning_rate, weight_decay=1e-4
        )
        return optimizer

    def on_train_epoch_end(self):
        # Store epoch metrics for plotting
        self.train_losses.append(np.mean(self.current_epoch_train_loss))
        self.train_accs.append(np.mean(self.current_epoch_train_acc))
        self.current_epoch_train_loss = []
        self.current_epoch_train_acc = []

    def on_validation_epoch_end(self):
        # Get the current validation metrics
        val_loss = self.trainer.callback_metrics["val_loss"].item()
        val_acc = self.trainer.callback_metrics["val_acc"].item()

        # Store for plotting
        self.val_losses.append(val_loss)
        self.val_accs.append(val_acc)

### Train the Model

Now, let's train the TSLANet model:

In [ ]:
# Define model parameters
embed_dim = 64
depth = 2
patch_size = 8
dropout_rate = 0.1
learning_rate = 1e-3

# Create TSLANet model
tslanet = TSLANet(
    seq_len=seq_len,
    num_classes=num_classes,
    num_channels=num_channels,
    embed_dim=embed_dim,
    depth=depth,
    patch_size=patch_size,
    dropout_rate=dropout_rate,
    learning_rate=learning_rate,
    use_asb=True,
    use_icb=True,
    adaptive_filter=True,
)

# Wrap the model for training
training_model = TSLANetTrainingWrapper(tslanet, learning_rate=learning_rate)

# Setup callbacks
checkpoint_dir = "./checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_callback = ModelCheckpoint(
    dirpath=checkpoint_dir,
    filename="tslanet-{epoch:02d}-{val_acc:.4f}",
    monitor="val_acc",
    mode="max",
    save_top_k=1,
)

early_stopping = EarlyStopping(monitor="val_acc", patience=10, mode="max", verbose=True)

# Setup trainer
trainer = L.Trainer(
    max_epochs=30,
    callbacks=[checkpoint_callback, early_stopping],
    accelerator="auto",
    devices=1,
    deterministic=True,
    enable_progress_bar=True,
    enable_model_summary=True,
)

# Train the model
trainer.fit(training_model, train_loader, val_loader)

### Plot Training Progress

Let's visualize the training progress:

In [ ]:
# Plot loss curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(training_model.train_losses, label="Training Loss")
plt.plot(training_model.val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(training_model.train_accs, label="Training Accuracy")
plt.plot(training_model.val_accs, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 3. Evaluating Model Performance

Now let's evaluate the model on the test set:

In [ ]:
# Test the model
test_results = trainer.test(training_model, test_loader)
print(f"Test results: {test_results}")

# Get predictions on test set
training_model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        x, y = batch
        logits = training_model(x)
        preds = logits.argmax(dim=1)

        y_true.extend(y.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Generate classification report
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print("Classification Report:")
print(report)

### Plot Confusion Matrix

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 4. Analyzing Model Components

Let's analyze the adaptive spectral filtering behavior of the model:

In [ ]:
def visualize_spectral_block(model, input_data, sample_idx=0, class_idx=None):
    """Visualize the Adaptive Spectral Block operation"""
    model.eval()

    # If class_idx is provided, find a sample from that class
    if class_idx is not None:
        for batch in test_loader:
            x, y = batch
            indices = torch.where(y == class_idx)[0]
            if len(indices) > 0:
                sample_idx = indices[0].item()
                input_data = x[indices[0:1]]
                break
    else:
        # Use the provided input_data and sample_idx
        input_data = input_data[sample_idx : sample_idx + 1]

    with torch.no_grad():
        # Get intermediate outputs
        # 1. Patch embedding
        patches = model.model.patch_embed(input_data)
        patches = patches + model.model.pos_embed
        patches = model.model.pos_drop(patches)

        # 2. Apply layer norm
        norm_patches = model.model.layers[0].norm1(patches)

        # 3. Get ASB
        asb = model.model.layers[0].asb

        # 4. Apply FFT
        patches_t = norm_patches.transpose(1, 2)  # [B, C, N]
        fft_patches = torch.fft.rfft(patches_t, dim=2, norm="ortho")
        fft_magnitude = torch.abs(fft_patches)[
            0, :, :
        ]  # First batch, all channels, all frequencies

        # 5. Get adaptive mask
        mask = (
            asb.create_adaptive_high_freq_mask(fft_patches)[0, :, 0].cpu().numpy()
        )  # First batch, all frequencies

        # 6. Get weights
        weight = torch.view_as_complex(asb.complex_weight).abs().cpu().numpy()
        weight_high = torch.view_as_complex(asb.complex_weight_high).abs().cpu().numpy()

        # 7. Apply ASB
        asb_output = asb(norm_patches)

        # 8. Convert to numpy for plotting
        input_signal = input_data[0, 0].cpu().numpy()  # First batch, first channel
        fft_magnitude = fft_magnitude[0].cpu().numpy()  # First channel

        # 9. Return all components for analysis or visualization
        return {
            "input_signal": input_signal,
            "patches": patches[0].cpu().numpy(),
            "norm_patches": norm_patches[0].cpu().numpy(),
            "fft_magnitude": fft_magnitude,
            "mask": mask,
            "weight": weight,
            "weight_high": weight_high,
            "asb_output": asb_output[0].cpu().numpy(),
        }


# Example usage
sample_components = visualize_spectral_block(training_model, x_sample, class_idx=0)
print("Components extracted from the Adaptive Spectral Block:")
for key, value in sample_components.items():
    if isinstance(value, np.ndarray):
        print(f"{key}: shape {value.shape}")
    else:
        print(f"{key}: {value}")